# Crude Oil Price Prediction (WTI & Brent)

> Production-ready Google Colab notebook for **next-day** and **next-7-day** forecasting using a stacked ensemble:
- LSTM (temporal patterns)
- XGBoost + LightGBM (feature-based learners)
- Ridge Regression meta-learner (stacking)

This notebook fetches and merges:
- EIA fundamentals
- FRED macro indicators
- yfinance market data
- NewsAPI sentiment (VADER)

> Update API keys in the configuration cell before running.

In [ ]:
# Colab/Notebook dependency installation
!pip -q install yfinance fredapi xgboost lightgbm newsapi-python vaderSentiment tensorflow scikit-learn joblib requests matplotlib seaborn

## 1) Configuration

> Add your API keys below. Keep placeholders if you only want to validate notebook structure.

- `EIA_API_KEY`: EIA API key
- `FRED_API_KEY`: FRED API key
- `NEWS_API_KEY`: NewsAPI key

In [ ]:
import os
import sys
import time
import json
import random
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import requests
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from fredapi import Fred
from newsapi import NewsApiClient
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import lightgbm as lgb

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# API keys (replace placeholders or set environment variables)
EIA_API_KEY = os.getenv("EIA_API_KEY", "YOUR_EIA_KEY")
FRED_API_KEY = os.getenv("FRED_API_KEY", "YOUR_FRED_KEY")
NEWS_API_KEY = os.getenv("NEWS_API_KEY", "YOUR_NEWS_KEY")

# Date range (past 5 years)
END_DATE = pd.Timestamp.today().normalize()
START_DATE = (END_DATE - pd.DateOffset(years=5)).normalize()

# Paths for cache/models (Colab-first with local fallback)
try:
    os.makedirs("/content/models", exist_ok=True)
    os.makedirs("/content/cache", exist_ok=True)
    MODEL_DIR = "/content/models"
    CACHE_DIR = "/content/cache"
except Exception:
    local_root = os.path.join(os.getcwd(), "colab_artifacts")
    MODEL_DIR = os.path.join(local_root, "models")
    CACHE_DIR = os.path.join(local_root, "cache")
    os.makedirs(MODEL_DIR, exist_ok=True)
    os.makedirs(CACHE_DIR, exist_ok=True)

print(f"START_DATE: {START_DATE.date()} | END_DATE: {END_DATE.date()}")
print(f"CACHE_DIR: {CACHE_DIR}")
print(f"MODEL_DIR: {MODEL_DIR}")

# Required source definitions
EIA_SPOT_SERIES = {"RWTC": "eia_wti_spot", "RBRTE": "eia_brent_spot"}
EIA_INVENTORY_SERIES = "PET.WCRSTUS1.W"
EIA_RIG_COUNT_CANDIDATES = [
    "PET.E_ERTRRO_XR0_NUS_C",
    "PET.E_ERTRRO_XR0_NUS_M",
    "PET.E_ERTRRO_XR0_NUS_D",
]

FRED_SERIES = {
    "DGS10": "fred_dgs10",
    "DEXUSEU": "fred_dexuseu",
    "DCOILWTICO": "fred_dcoilwtico",
    "CPIAUCSL": "fred_cpi",
    "UNRATE": "fred_unrate",
}

YF_TICKERS = ["CL=F", "BZ=F", "DX-Y.NYB", "GC=F", "^GSPC"]
NEWS_QUERY = '"crude oil" OR "OPEC" OR "petroleum"'

# Runtime configuration (tunable for Colab free tier)
SEQUENCE_LENGTH = 30
STACKING_SPLITS = 3
LSTM_EPOCHS_MAIN = 80
LSTM_EPOCHS_OOF = 35
BATCH_SIZE = 32

## 2) Utility Functions

> Helpers for caching, safe API calls, and standard date/index processing.

In [ ]:
def is_placeholder_key(key: str) -> bool:
    if key is None:
        return True
    key = str(key).strip()
    return key == "" or key.startswith("YOUR_")


def cache_file(name: str) -> str:
    return os.path.join(CACHE_DIR, f"{name}.csv")


def load_cached_dataframe(name: str, parse_dates=True) -> pd.DataFrame | None:
    path = cache_file(name)
    if not os.path.exists(path):
        return None
    try:
        df = pd.read_csv(path)
        if parse_dates and "date" in df.columns:
            df["date"] = pd.to_datetime(df["date"], errors="coerce")
        print(f"Loaded cache: {path} ({len(df):,} rows)")
        return df
    except Exception as exc:
        print(f"[CACHE READ ERROR] {name}: {exc}")
        return None


def save_cached_dataframe(df: pd.DataFrame, name: str) -> None:
    path = cache_file(name)
    try:
        to_save = df.copy()
        if isinstance(to_save.index, pd.DatetimeIndex):
            to_save = to_save.reset_index().rename(columns={to_save.index.name or "index": "date"})
        to_save.to_csv(path, index=False)
        print(f"Saved cache: {path} ({len(to_save):,} rows)")
    except Exception as exc:
        print(f"[CACHE WRITE ERROR] {name}: {exc}")


def request_json(url: str, params: dict | None = None, headers: dict | None = None, timeout: int = 30, retries: int = 3):
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=timeout)
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            last_error = exc
            sleep_for = min(2 ** attempt, 8)
            print(f"[HTTP RETRY] attempt={attempt}/{retries} | url={url} | error={exc}")
            time.sleep(sleep_for)
    raise RuntimeError(f"Request failed after retries: {url} | error={last_error}")


def to_daily_index(df: pd.DataFrame, date_col: str = "date") -> pd.DataFrame:
    out = df.copy()
    if date_col in out.columns:
        out[date_col] = pd.to_datetime(out[date_col], errors="coerce")
        out = out.dropna(subset=[date_col]).set_index(date_col)
    if not isinstance(out.index, pd.DatetimeIndex):
        out.index = pd.to_datetime(out.index, errors="coerce")
    out = out.sort_index()
    out = out[~out.index.duplicated(keep="last")]

    full_range = pd.date_range(START_DATE, END_DATE, freq="D")
    out = out.reindex(full_range)
    out.index.name = "date"
    return out


def print_df_info(name: str, df: pd.DataFrame):
    print(f"[{name}] shape={df.shape}")
    if len(df) > 0:
        print(f"[{name}] range={df.index.min().date()} -> {df.index.max().date()}")
    print("-" * 60)

## 3) Data Ingestion (EIA, FRED, yfinance, NewsAPI)

> Each source has independent `try/except`, CSV cache, and schema normalization before merge.

In [ ]:
def fetch_eia_spot_prices(force_refresh: bool = False) -> pd.DataFrame:
    cache_name = "eia_spot_prices"
    if not force_refresh:
        cached = load_cached_dataframe(cache_name)
        if cached is not None and len(cached) > 0:
            return to_daily_index(cached)

    if is_placeholder_key(EIA_API_KEY):
        print("[EIA] Placeholder API key detected. Returning empty EIA spot DataFrame.")
        return to_daily_index(pd.DataFrame(columns=["date", "eia_wti_spot", "eia_brent_spot"]))

    all_parts = []
    for series_code, col_name in EIA_SPOT_SERIES.items():
        try:
            url = "https://api.eia.gov/v2/petroleum/pri/spt/data/"
            params = {
                "api_key": EIA_API_KEY,
                "frequency": "daily",
                "data[0]": "value",
                "facets[series][]": series_code,
                "start": START_DATE.strftime("%Y-%m-%d"),
                "end": END_DATE.strftime("%Y-%m-%d"),
                "sort[0][column]": "period",
                "sort[0][direction]": "asc",
                "offset": 0,
                "length": 5000,
            }
            payload = request_json(url, params=params)
            records = payload.get("response", {}).get("data", [])
            if len(records) == 0:
                print(f"[EIA] No records for series {series_code}")
                continue

            temp = pd.DataFrame(records)
            period_col = "period" if "period" in temp.columns else "date"
            value_col = "value" if "value" in temp.columns else temp.columns[-1]
            temp = temp[[period_col, value_col]].rename(columns={period_col: "date", value_col: col_name})
            temp["date"] = pd.to_datetime(temp["date"], errors="coerce")
            temp[col_name] = pd.to_numeric(temp[col_name], errors="coerce")
            temp = temp.dropna(subset=["date"]).sort_values("date")
            all_parts.append(temp)
            print(f"[EIA] Fetched {len(temp):,} rows for {series_code}")
        except Exception as exc:
            print(f"[EIA ERROR] spot series {series_code}: {exc}")

    if len(all_parts) == 0:
        return to_daily_index(pd.DataFrame(columns=["date", "eia_wti_spot", "eia_brent_spot"]))

    spot_df = all_parts[0]
    for part in all_parts[1:]:
        spot_df = spot_df.merge(part, on="date", how="outer")
    spot_df = to_daily_index(spot_df)
    save_cached_dataframe(spot_df, cache_name)
    return spot_df


def fetch_eia_legacy_series(series_id: str, value_name: str, force_refresh: bool = False) -> pd.DataFrame:
    cache_name = f"eia_legacy_{series_id.replace('.', '_')}"
    if not force_refresh:
        cached = load_cached_dataframe(cache_name)
        if cached is not None and len(cached) > 0:
            return to_daily_index(cached)

    if is_placeholder_key(EIA_API_KEY):
        print(f"[EIA] Placeholder API key detected. Returning empty DataFrame for {series_id}.")
        return to_daily_index(pd.DataFrame(columns=["date", value_name]))

    try:
        url = "https://api.eia.gov/series/"
        params = {"api_key": EIA_API_KEY, "series_id": series_id}
        payload = request_json(url, params=params)
        series = payload.get("series", [])
        if len(series) == 0 or "data" not in series[0]:
            raise ValueError(f"No series data returned for {series_id}")

        records = series[0]["data"]
        temp = pd.DataFrame(records, columns=["date", value_name])
        temp["date"] = pd.to_datetime(temp["date"].astype(str), errors="coerce")
        temp[value_name] = pd.to_numeric(temp[value_name], errors="coerce")
        temp = temp.dropna(subset=["date"]).sort_values("date")
        temp = to_daily_index(temp)
        save_cached_dataframe(temp, cache_name)
        return temp
    except Exception as exc:
        print(f"[EIA ERROR] series {series_id}: {exc}")
        return to_daily_index(pd.DataFrame(columns=["date", value_name]))


def fetch_eia_dataset(force_refresh: bool = False) -> pd.DataFrame:
    spot_df = fetch_eia_spot_prices(force_refresh=force_refresh)

    inv_df = fetch_eia_legacy_series(
        series_id=EIA_INVENTORY_SERIES,
        value_name="eia_crude_inventory",
        force_refresh=force_refresh,
    )

    rig_df = None
    for candidate in EIA_RIG_COUNT_CANDIDATES:
        temp = fetch_eia_legacy_series(
            series_id=candidate,
            value_name="eia_rig_count",
            force_refresh=force_refresh,
        )
        if temp["eia_rig_count"].notna().sum() > 0:
            rig_df = temp
            print(f"[EIA] Using rig count series: {candidate}")
            break
    if rig_df is None:
        print("[EIA] Rig count not available via provided candidates. Filling with NaN.")
        rig_df = to_daily_index(pd.DataFrame(columns=["date", "eia_rig_count"]))

    eia_df = spot_df.join(inv_df, how="outer").join(rig_df, how="outer")
    eia_df = to_daily_index(eia_df)
    eia_df = eia_df.ffill()
    print_df_info("EIA", eia_df)
    return eia_df


eia_df = fetch_eia_dataset(force_refresh=False)

In [ ]:
def fetch_fred_dataset(force_refresh: bool = False) -> pd.DataFrame:
    cache_name = "fred_dataset"
    if not force_refresh:
        cached = load_cached_dataframe(cache_name)
        if cached is not None and len(cached) > 0:
            return to_daily_index(cached).ffill()

    if is_placeholder_key(FRED_API_KEY):
        print("[FRED] Placeholder API key detected. Returning empty FRED DataFrame.")
        return to_daily_index(pd.DataFrame(columns=["date"] + list(FRED_SERIES.values())))

    try:
        fred = Fred(api_key=FRED_API_KEY)
    except Exception as exc:
        print(f"[FRED ERROR] Unable to initialize Fred client: {exc}")
        return to_daily_index(pd.DataFrame(columns=["date"] + list(FRED_SERIES.values())))

    merged = None
    for series_id, col_name in FRED_SERIES.items():
        try:
            series = fred.get_series(
                series_id,
                observation_start=START_DATE.strftime("%Y-%m-%d"),
                observation_end=END_DATE.strftime("%Y-%m-%d"),
            )
            part = series.rename(col_name).to_frame()
            part.index = pd.to_datetime(part.index, errors="coerce")
            part = part.dropna().sort_index()
            if merged is None:
                merged = part
            else:
                merged = merged.join(part, how="outer")
            print(f"[FRED] Fetched {series_id} ({len(part):,} rows)")
        except Exception as exc:
            print(f"[FRED ERROR] series {series_id}: {exc}")

    if merged is None:
        merged = pd.DataFrame(index=pd.date_range(START_DATE, END_DATE, freq="D"))

    merged.index.name = "date"
    merged = to_daily_index(merged)
    merged = merged.ffill()
    save_cached_dataframe(merged, cache_name)
    print_df_info("FRED", merged)
    return merged


fred_df = fetch_fred_dataset(force_refresh=False)

In [ ]:
def clean_ticker_name(ticker: str) -> str:
    return "".join(ch if ch.isalnum() else "_" for ch in ticker)


def fetch_yfinance_dataset(force_refresh: bool = False) -> pd.DataFrame:
    cache_name = "yfinance_dataset"
    if not force_refresh:
        cached = load_cached_dataframe(cache_name)
        if cached is not None and len(cached) > 0:
            return to_daily_index(cached).ffill()

    try:
        raw = yf.download(
            YF_TICKERS,
            start=START_DATE.strftime("%Y-%m-%d"),
            end=(END_DATE + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
            interval="1d",
            auto_adjust=False,
            progress=False,
            group_by="ticker",
            threads=True,
        )
        if raw is None or len(raw) == 0:
            raise ValueError("Empty response from yfinance")

        parts = []
        if isinstance(raw.columns, pd.MultiIndex):
            for ticker in YF_TICKERS:
                if ticker not in raw.columns.get_level_values(0):
                    print(f"[YFINANCE] Missing ticker in response: {ticker}")
                    continue
                temp = raw[ticker].copy()
                temp.columns = [f"{clean_ticker_name(ticker)}_{str(c).lower()}" for c in temp.columns]
                parts.append(temp)
        else:
            temp = raw.copy()
            temp.columns = [f"{clean_ticker_name(YF_TICKERS[0])}_{str(c).lower()}" for c in temp.columns]
            parts.append(temp)

        if len(parts) == 0:
            raise ValueError("No ticker data parsed from yfinance response")

        yf_df = pd.concat(parts, axis=1)
        yf_df.index = pd.to_datetime(yf_df.index, errors="coerce")
        yf_df = yf_df.sort_index()
        yf_df = to_daily_index(yf_df)
        yf_df = yf_df.ffill()

        save_cached_dataframe(yf_df, cache_name)
        print_df_info("YFINANCE", yf_df)
        return yf_df
    except Exception as exc:
        print(f"[YFINANCE ERROR] {exc}")
        empty_cols = []
        for ticker in YF_TICKERS:
            safe = clean_ticker_name(ticker)
            empty_cols.extend([
                f"{safe}_open",
                f"{safe}_high",
                f"{safe}_low",
                f"{safe}_close",
                f"{safe}_adj close",
                f"{safe}_volume",
            ])
        return to_daily_index(pd.DataFrame(columns=["date"] + empty_cols))


yf_df = fetch_yfinance_dataset(force_refresh=False)

In [ ]:
def fetch_news_sentiment_dataset(force_refresh: bool = False, max_pages: int = 20) -> pd.DataFrame:
    cache_name = "newsapi_sentiment"
    if not force_refresh:
        cached = load_cached_dataframe(cache_name)
        if cached is not None and len(cached) > 0:
            out = to_daily_index(cached)
            if "news_sentiment" in out.columns:
                out["news_sentiment"] = out["news_sentiment"].ffill().fillna(0.0)
            if "news_article_count" in out.columns:
                out["news_article_count"] = out["news_article_count"].ffill().fillna(0.0)
            return out

    empty = to_daily_index(pd.DataFrame(columns=["date", "news_sentiment", "news_article_count"]))
    if is_placeholder_key(NEWS_API_KEY):
        print("[NEWSAPI] Placeholder API key detected. Returning empty sentiment DataFrame.")
        empty["news_sentiment"] = 0.0
        empty["news_article_count"] = 0.0
        return empty

    try:
        client = NewsApiClient(api_key=NEWS_API_KEY)
        analyzer = SentimentIntensityAnalyzer()
    except Exception as exc:
        print(f"[NEWSAPI ERROR] Client initialization failed: {exc}")
        empty["news_sentiment"] = 0.0
        empty["news_article_count"] = 0.0
        return empty

    rows = []
    try:
        for page in range(1, max_pages + 1):
            result = client.get_everything(
                q=NEWS_QUERY,
                from_param=START_DATE.strftime("%Y-%m-%d"),
                to=END_DATE.strftime("%Y-%m-%d"),
                language="en",
                sort_by="publishedAt",
                page_size=100,
                page=page,
            )
            articles = result.get("articles", [])
            if len(articles) == 0:
                break

            for art in articles:
                title = (art.get("title") or "").strip()
                published_at = art.get("publishedAt")
                if published_at is None or title == "":
                    continue
                dt = pd.to_datetime(published_at, errors="coerce")
                if pd.isna(dt):
                    continue
                score = analyzer.polarity_scores(title).get("compound", 0.0)
                rows.append({"date": dt.normalize(), "compound": score})

            total_results = result.get("totalResults", 0)
            if page * 100 >= total_results:
                break
            time.sleep(0.25)

        if len(rows) == 0:
            print("[NEWSAPI] No articles retrieved. Using default zeros.")
            empty["news_sentiment"] = 0.0
            empty["news_article_count"] = 0.0
            return empty

        sentiment = pd.DataFrame(rows)
        sentiment = sentiment.groupby("date", as_index=False).agg(
            news_sentiment=("compound", "mean"),
            news_article_count=("compound", "size"),
        )
        sentiment = to_daily_index(sentiment)
        sentiment["news_sentiment"] = sentiment["news_sentiment"].ffill().fillna(0.0)
        sentiment["news_article_count"] = sentiment["news_article_count"].ffill().fillna(0.0)

        save_cached_dataframe(sentiment, cache_name)
        print_df_info("NEWS_SENTIMENT", sentiment)
        return sentiment
    except Exception as exc:
        print(f"[NEWSAPI ERROR] Fetch failed: {exc}")
        empty["news_sentiment"] = 0.0
        empty["news_article_count"] = 0.0
        return empty


news_df = fetch_news_sentiment_dataset(force_refresh=False, max_pages=20)

## 4) Unified Daily DataFrame

> Merge all sources on date index and derive canonical targets (`WTI_close`, `Brent_close`).

In [ ]:
def build_unified_daily_df(
    eia_data: pd.DataFrame,
    fred_data: pd.DataFrame,
    yf_data: pd.DataFrame,
    news_data: pd.DataFrame,
) -> pd.DataFrame:
    merged = eia_data.join(fred_data, how="outer")
    merged = merged.join(yf_data, how="outer")
    merged = merged.join(news_data, how="outer")
    merged = to_daily_index(merged)
    merged = merged.sort_index().ffill()

    for needed_col in ["eia_wti_spot", "eia_brent_spot", "fred_dcoilwtico", "CL_F_close", "BZ_F_close"]:
        if needed_col not in merged.columns:
            merged[needed_col] = np.nan

    # Canonical targets with fallback order
    merged["WTI_close"] = merged["eia_wti_spot"]
    merged["WTI_close"] = merged["WTI_close"].combine_first(merged["CL_F_close"])
    merged["WTI_close"] = merged["WTI_close"].combine_first(merged["fred_dcoilwtico"])

    merged["Brent_close"] = merged["eia_brent_spot"]
    merged["Brent_close"] = merged["Brent_close"].combine_first(merged["BZ_F_close"])

    merged = merged.ffill()
    merged = merged.dropna(subset=["WTI_close", "Brent_close"])
    return merged


df_raw = build_unified_daily_df(eia_df, fred_df, yf_df, news_df)
print_df_info("UNIFIED", df_raw)
display(df_raw.tail(5))

## 5) Feature Engineering & Dual Pipelines

> Pipeline A (LSTM): lag/rolling features + MinMax scaling + sequence windowing.

- `price_lag_1`, `price_lag_3`, `price_lag_7`
- `rolling_mean_7`, `rolling_std_7`, `rolling_mean_14`

> Pipeline B (GBM): technical indicators + calendar features + Standard scaling.

- RSI(14), MACD, Bollinger Bands, ATR
- day_of_week, month, is_month_end

In [ ]:
def compute_rsi(series: pd.Series, window: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / window, min_periods=window, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / window, min_periods=window, adjust=False).mean()
    rs = avg_gain / (avg_loss + 1e-9)
    return 100 - (100 / (1 + rs))


def compute_macd(series: pd.Series):
    ema12 = series.ewm(span=12, adjust=False).mean()
    ema26 = series.ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    signal = macd.ewm(span=9, adjust=False).mean()
    hist = macd - signal
    return macd, signal, hist


def compute_bollinger(series: pd.Series, window: int = 20, n_std: float = 2.0):
    ma = series.rolling(window).mean()
    std = series.rolling(window).std()
    upper = ma + n_std * std
    lower = ma - n_std * std
    return ma, upper, lower


def compute_atr(high: pd.Series, low: pd.Series, close: pd.Series, window: int = 14) -> pd.Series:
    prev_close = close.shift(1)
    tr = pd.concat([
        (high - low),
        (high - prev_close).abs(),
        (low - prev_close).abs(),
    ], axis=1).max(axis=1)
    return tr.rolling(window).mean()


def add_technical_and_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # WTI futures technicals from CL=F
    if all(c in out.columns for c in ["CL_F_close", "CL_F_high", "CL_F_low"]):
        out["wti_rsi14"] = compute_rsi(out["CL_F_close"], 14)
        macd, signal, hist = compute_macd(out["CL_F_close"] )
        out["wti_macd"] = macd
        out["wti_macd_signal"] = signal
        out["wti_macd_hist"] = hist
        bb_mid, bb_up, bb_low = compute_bollinger(out["CL_F_close"], window=20, n_std=2.0)
        out["wti_bb_mid"] = bb_mid
        out["wti_bb_up"] = bb_up
        out["wti_bb_low"] = bb_low
        out["wti_atr14"] = compute_atr(out["CL_F_high"], out["CL_F_low"], out["CL_F_close"], window=14)

    # Brent futures technicals from BZ=F
    if all(c in out.columns for c in ["BZ_F_close", "BZ_F_high", "BZ_F_low"]):
        out["brent_rsi14"] = compute_rsi(out["BZ_F_close"], 14)
        macd, signal, hist = compute_macd(out["BZ_F_close"] )
        out["brent_macd"] = macd
        out["brent_macd_signal"] = signal
        out["brent_macd_hist"] = hist
        bb_mid, bb_up, bb_low = compute_bollinger(out["BZ_F_close"], window=20, n_std=2.0)
        out["brent_bb_mid"] = bb_mid
        out["brent_bb_up"] = bb_up
        out["brent_bb_low"] = bb_low
        out["brent_atr14"] = compute_atr(out["BZ_F_high"], out["BZ_F_low"], out["BZ_F_close"], window=14)

    # Calendar features
    out["day_of_week"] = out.index.dayofweek
    out["month"] = out.index.month
    out["is_month_end"] = out.index.is_month_end.astype(int)
    return out


def build_modeling_frame(base_df: pd.DataFrame, target_col: str, horizon: int) -> pd.DataFrame:
    df = base_df.copy()

    # Required LSTM lag/rolling feature names
    df["price_lag_1"] = df[target_col].shift(1)
    df["price_lag_3"] = df[target_col].shift(3)
    df["price_lag_7"] = df[target_col].shift(7)
    df["rolling_mean_7"] = df[target_col].rolling(7).mean()
    df["rolling_std_7"] = df[target_col].rolling(7).std()
    df["rolling_mean_14"] = df[target_col].rolling(14).mean()

    # GBM-specific feature enrichment
    df = add_technical_and_calendar_features(df)

    # Forecast target (t + horizon)
    df["target"] = df[target_col].shift(-horizon)

    # Missing handling as required
    df = df.ffill()
    df = df.dropna()
    return df


def make_supervised_views(
    model_df: pd.DataFrame,
    target_col: str,
    sequence_length: int = 30,
 ) -> dict:
    numeric_cols = model_df.select_dtypes(include=[np.number]).columns.tolist()
    if "target" not in numeric_cols:
        raise ValueError("Target column missing from modeling frame.")

    # LSTM and GBM both use numeric features (excluding target output)
    feature_cols = [c for c in numeric_cols if c != "target"]
    lstm_feature_cols = feature_cols.copy()
    gbm_feature_cols = feature_cols.copy()

    X_seq, X_flat, y, base_price = [], [], [], []
    origin_dates = []
    for i in range(sequence_length, len(model_df)):
        past_window = model_df.iloc[i - sequence_length:i]
        X_seq.append(past_window[lstm_feature_cols].values.astype(np.float32))
        X_flat.append(model_df.iloc[i][gbm_feature_cols].values.astype(np.float32))
        y.append(float(model_df.iloc[i]["target"]))
        base_price.append(float(model_df.iloc[i][target_col]))
        origin_dates.append(model_df.index[i])

    result = {
        "X_seq": np.array(X_seq, dtype=np.float32),
        "X_flat": np.array(X_flat, dtype=np.float32),
        "y": np.array(y, dtype=np.float32),
        "base_price": np.array(base_price, dtype=np.float32),
        "origin_dates": pd.to_datetime(origin_dates),
        "lstm_feature_cols": lstm_feature_cols,
        "gbm_feature_cols": gbm_feature_cols,
    }
    return result


def chronological_split_indices(n_samples: int, train_ratio: float = 0.70, val_ratio: float = 0.15):
    train_end = int(n_samples * train_ratio)
    val_end = int(n_samples * (train_ratio + val_ratio))
    return train_end, val_end

## 6) Modeling: LSTM, XGBoost, LightGBM, and Stacking

> Chronological split: **70% train / 15% validation / 15% test** (no shuffle).

The stacker uses out-of-fold level-1 predictions from all three base models and trains a Ridge meta-learner.

In [ ]:
def directional_accuracy(y_true: np.ndarray, y_pred: np.ndarray, base_price: np.ndarray) -> float:
    actual_dir = np.sign(y_true - base_price)
    pred_dir = np.sign(y_pred - base_price)
    return float(np.mean(actual_dir == pred_dir))


def compute_regression_metrics(y_true: np.ndarray, y_pred: np.ndarray, base_price: np.ndarray) -> dict:
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    denom = np.where(np.abs(y_true) < 1e-8, 1e-8, np.abs(y_true))
    mape = np.mean(np.abs((y_true - y_pred) / denom)) * 100
    r2 = r2_score(y_true, y_pred)
    dir_acc = directional_accuracy(y_true, y_pred, base_price) * 100
    return {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "MAPE": float(mape),
        "R2": float(r2),
        "Dir Accuracy": float(dir_acc),
    }


def plot_last_90_days(
    dates: pd.DatetimeIndex,
    y_true: np.ndarray,
    preds: dict,
    title: str,
    last_n: int = 90,
 ):
    take = min(last_n, len(y_true))
    idx = np.arange(len(y_true) - take, len(y_true))
    plt.figure(figsize=(14, 6))
    plt.plot(dates[idx], y_true[idx], label="Actual", linewidth=2.5, color="black")
    for name, arr in preds.items():
        plt.plot(dates[idx], arr[idx], label=name, linewidth=1.8)
    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_xgb_feature_importance(model: XGBRegressor, feature_names: list[str], title: str, top_n: int = 20):
    importances = getattr(model, "feature_importances_", None)
    if importances is None or len(importances) == 0:
        print("[XGBOOST] Feature importances not available.")
        return
    order = np.argsort(importances)[::-1][:top_n]
    fi_df = pd.DataFrame({
        "feature": np.array(feature_names)[order],
        "importance": importances[order],
    })
    plt.figure(figsize=(10, 6))
    sns.barplot(data=fi_df, x="importance", y="feature", orient="h")
    plt.title(title)
    plt.tight_layout()
    plt.show()


def build_lstm_model(input_shape: tuple) -> tf.keras.Model:
    model = Sequential([
        LSTM(128, return_sequences=True, dropout=0.2, input_shape=input_shape),
        LSTM(64, return_sequences=False, dropout=0.2),
        Dense(32, activation="relu"),
        Dense(1),
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss=tf.keras.losses.Huber(),
    )
    return model


def scale_lstm_sequences_fit(X_seq_train: np.ndarray, y_train: np.ndarray):
    n, t, f = X_seq_train.shape
    feat_scaler = MinMaxScaler()
    X_train_2d = X_seq_train.reshape(-1, f)
    X_train_scaled = feat_scaler.fit_transform(X_train_2d).reshape(n, t, f)

    y_scaler = MinMaxScaler()
    y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
    return feat_scaler, y_scaler, X_train_scaled, y_train_scaled


def scale_lstm_sequences_transform(X_seq: np.ndarray, feat_scaler: MinMaxScaler) -> np.ndarray:
    n, t, f = X_seq.shape
    X_2d = X_seq.reshape(-1, f)
    X_scaled = feat_scaler.transform(X_2d).reshape(n, t, f)
    return X_scaled


def train_lstm_single_split(
    X_seq_train: np.ndarray,
    y_train: np.ndarray,
    X_seq_val: np.ndarray,
    y_val: np.ndarray,
    epochs: int,
 ):
    feat_scaler, y_scaler, X_train_scaled, y_train_scaled = scale_lstm_sequences_fit(X_seq_train, y_train)
    X_val_scaled = scale_lstm_sequences_transform(X_seq_val, feat_scaler)
    y_val_scaled = y_scaler.transform(y_val.reshape(-1, 1)).ravel()

    model = build_lstm_model(input_shape=(X_train_scaled.shape[1], X_train_scaled.shape[2]))
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5),
    ]
    model.fit(
        X_train_scaled,
        y_train_scaled,
        validation_data=(X_val_scaled, y_val_scaled),
        epochs=epochs,
        batch_size=BATCH_SIZE,
        verbose=0,
        callbacks=callbacks,
    )
    return model, feat_scaler, y_scaler


def train_xgb_single_split(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
 ):
    model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=SEED,
    )
    try:
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=False,
            early_stopping_rounds=30,
        )
    except Exception as exc:
        print(f"[XGBOOST WARNING] Early stopping unavailable: {exc}")
        model.fit(X_train, y_train)
    return model


def train_lgbm_single_split(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
 ):
    model = LGBMRegressor(
        objective="regression",
        metric="rmse",
        num_leaves=63,
        learning_rate=0.05,
        n_estimators=500,
        random_state=SEED,
    )
    try:
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            callbacks=[lgb.early_stopping(30, verbose=False)],
        )
    except Exception as exc:
        print(f"[LIGHTGBM WARNING] Early stopping unavailable: {exc}")
        model.fit(X_train, y_train)
    return model


def build_level1_oof_predictions(
    X_seq_tv: np.ndarray,
    X_flat_tv: np.ndarray,
    y_tv: np.ndarray,
    n_splits: int = 3,
 ):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    n = len(y_tv)
    oof = np.full((n, 3), np.nan, dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_flat_tv), start=1):
        print(f"[OOF] Fold {fold}/{n_splits} | train={len(tr_idx):,} val={len(va_idx):,}")
        if len(tr_idx) < 100 or len(va_idx) < 20:
            print("[OOF] Fold skipped due to insufficient samples.")
            continue

        X_seq_tr, X_seq_va = X_seq_tv[tr_idx], X_seq_tv[va_idx]
        X_flat_tr, X_flat_va = X_flat_tv[tr_idx], X_flat_tv[va_idx]
        y_tr, y_va = y_tv[tr_idx], y_tv[va_idx]

        # LSTM fold
        lstm_model, lstm_feat_scaler, lstm_target_scaler = train_lstm_single_split(
            X_seq_tr, y_tr, X_seq_va, y_va, epochs=LSTM_EPOCHS_OOF
        )
        X_va_scaled = scale_lstm_sequences_transform(X_seq_va, lstm_feat_scaler)
        pred_lstm_scaled = lstm_model.predict(X_va_scaled, verbose=0).ravel()
        pred_lstm = lstm_target_scaler.inverse_transform(pred_lstm_scaled.reshape(-1, 1)).ravel()

        # GBM fold
        gbm_scaler = StandardScaler()
        X_tr_g = gbm_scaler.fit_transform(X_flat_tr)
        X_va_g = gbm_scaler.transform(X_flat_va)

        xgb_fold = train_xgb_single_split(X_tr_g, y_tr, X_va_g, y_va)
        lgb_fold = train_lgbm_single_split(X_tr_g, y_tr, X_va_g, y_va)

        pred_xgb = xgb_fold.predict(X_va_g).ravel()
        pred_lgb = lgb_fold.predict(X_va_g).ravel()

        oof[va_idx, 0] = pred_lstm
        oof[va_idx, 1] = pred_xgb
        oof[va_idx, 2] = pred_lgb

    valid_mask = ~np.isnan(oof).any(axis=1)
    if valid_mask.sum() == 0:
        raise RuntimeError("No valid OOF predictions generated for stacking.")
    return oof[valid_mask], y_tv[valid_mask], valid_mask


def save_task_artifacts(
    task_name: str,
    lstm_model: tf.keras.Model,
    xgb_model: XGBRegressor,
    lgbm_model: LGBMRegressor,
    ridge_model: Ridge,
    lstm_feature_scaler: MinMaxScaler,
    lstm_target_scaler: MinMaxScaler,
    gbm_scaler: StandardScaler,
    metadata: dict,
 ):
    os.makedirs(MODEL_DIR, exist_ok=True)
    lstm_path = os.path.join(MODEL_DIR, f"lstm_{task_name}.keras")
    xgb_path = os.path.join(MODEL_DIR, f"xgb_{task_name}.joblib")
    lgb_path = os.path.join(MODEL_DIR, f"lgbm_{task_name}.joblib")
    ridge_path = os.path.join(MODEL_DIR, f"ridge_{task_name}.joblib")
    lstm_feat_scaler_path = os.path.join(MODEL_DIR, f"lstm_feature_scaler_{task_name}.joblib")
    lstm_target_scaler_path = os.path.join(MODEL_DIR, f"lstm_target_scaler_{task_name}.joblib")
    gbm_scaler_path = os.path.join(MODEL_DIR, f"gbm_scaler_{task_name}.joblib")
    metadata_path = os.path.join(MODEL_DIR, f"metadata_{task_name}.joblib")

    lstm_model.save(lstm_path)
    joblib.dump(xgb_model, xgb_path)
    joblib.dump(lgbm_model, lgb_path)
    joblib.dump(ridge_model, ridge_path)
    joblib.dump(lstm_feature_scaler, lstm_feat_scaler_path)
    joblib.dump(lstm_target_scaler, lstm_target_scaler_path)
    joblib.dump(gbm_scaler, gbm_scaler_path)
    joblib.dump(metadata, metadata_path)

    print("Saved artifacts:")
    print(f"- {lstm_path}")
    print(f"- {xgb_path}")
    print(f"- {lgb_path}")
    print(f"- {ridge_path}")
    print(f"- {lstm_feat_scaler_path}")
    print(f"- {lstm_target_scaler_path}")
    print(f"- {gbm_scaler_path}")
    print(f"- {metadata_path}")


def run_single_experiment(base_df: pd.DataFrame, target_col: str, horizon: int) -> dict:
    print("=" * 80)
    print(f"Running experiment: target={target_col}, horizon={horizon} day(s)")
    model_df = build_modeling_frame(base_df, target_col=target_col, horizon=horizon)
    views = make_supervised_views(model_df, target_col=target_col, sequence_length=SEQUENCE_LENGTH)

    X_seq = views["X_seq"]
    X_flat = views["X_flat"]
    y = views["y"]
    base_price = views["base_price"]
    origin_dates = views["origin_dates"]
    target_dates = origin_dates + pd.Timedelta(days=horizon)

    n_samples = len(y)
    train_end, val_end = chronological_split_indices(n_samples)
    if train_end <= 0 or val_end <= train_end or val_end >= n_samples:
        raise ValueError("Insufficient samples for 70/15/15 split.")

    X_seq_train, X_seq_val, X_seq_test = X_seq[:train_end], X_seq[train_end:val_end], X_seq[val_end:]
    X_flat_train, X_flat_val, X_flat_test = X_flat[:train_end], X_flat[train_end:val_end], X_flat[val_end:]
    y_train, y_val, y_test = y[:train_end], y[train_end:val_end], y[val_end:]
    base_test = base_price[val_end:]
    dates_test = target_dates[val_end:]

    print(f"Samples: train={len(y_train):,} | val={len(y_val):,} | test={len(y_test):,}")

    # LSTM base model
    lstm_model, lstm_feature_scaler, lstm_target_scaler = train_lstm_single_split(
        X_seq_train, y_train, X_seq_val, y_val, epochs=LSTM_EPOCHS_MAIN
    )
    X_seq_test_scaled = scale_lstm_sequences_transform(X_seq_test, lstm_feature_scaler)
    pred_lstm_scaled = lstm_model.predict(X_seq_test_scaled, verbose=0).ravel()
    pred_lstm = lstm_target_scaler.inverse_transform(pred_lstm_scaled.reshape(-1, 1)).ravel()

    # GBM base models
    gbm_scaler = StandardScaler()
    X_train_g = gbm_scaler.fit_transform(X_flat_train)
    X_val_g = gbm_scaler.transform(X_flat_val)
    X_test_g = gbm_scaler.transform(X_flat_test)

    xgb_model = train_xgb_single_split(X_train_g, y_train, X_val_g, y_val)
    lgbm_model = train_lgbm_single_split(X_train_g, y_train, X_val_g, y_val)

    pred_xgb = xgb_model.predict(X_test_g).ravel()
    pred_lgbm = lgbm_model.predict(X_test_g).ravel()

    # OOF stacking on train+val
    X_seq_tv = X_seq[:val_end]
    X_flat_tv = X_flat[:val_end]
    y_tv = y[:val_end]
    oof_preds, y_oof, _ = build_level1_oof_predictions(
        X_seq_tv=X_seq_tv,
        X_flat_tv=X_flat_tv,
        y_tv=y_tv,
        n_splits=STACKING_SPLITS,
    )
    ridge_meta = Ridge(alpha=1.0)
    ridge_meta.fit(oof_preds, y_oof)

    level1_test = np.column_stack([pred_lstm, pred_xgb, pred_lgbm])
    pred_stack = ridge_meta.predict(level1_test).ravel()

    # Metrics
    metrics = {}
    metrics["LSTM"] = compute_regression_metrics(y_test, pred_lstm, base_test)
    metrics["XGBoost"] = compute_regression_metrics(y_test, pred_xgb, base_test)
    metrics["LightGBM"] = compute_regression_metrics(y_test, pred_lgbm, base_test)
    metrics["Stacked(Ridge)"] = compute_regression_metrics(y_test, pred_stack, base_test)

    results_df = pd.DataFrame(metrics).T.reset_index().rename(columns={"index": "Model"})
    results_df["Target"] = target_col
    results_df["Horizon"] = horizon
    results_df = results_df[["Target", "Horizon", "Model", "MAE", "RMSE", "MAPE", "R2", "Dir Accuracy"]]

    display(results_df.sort_values(by="RMSE"))

    # Visual diagnostics
    plot_last_90_days(
        dates=dates_test,
        y_true=y_test,
        preds={
            "LSTM": pred_lstm,
            "XGBoost": pred_xgb,
            "LightGBM": pred_lgbm,
            "Stacked(Ridge)": pred_stack,
        },
        title=f"Actual vs Predicted ({target_col}, horizon={horizon}d)",
        last_n=90,
    )

    plot_xgb_feature_importance(
        xgb_model,
        feature_names=views["gbm_feature_cols"],
        title=f"XGBoost Feature Importance ({target_col}, horizon={horizon}d)",
        top_n=20,
    )

    task_name = f"{target_col.lower()}_h{horizon}"
    metadata = {
        "target_col": target_col,
        "horizon": horizon,
        "sequence_length": SEQUENCE_LENGTH,
        "lstm_feature_cols": views["lstm_feature_cols"],
        "gbm_feature_cols": views["gbm_feature_cols"],
    }

    save_task_artifacts(
        task_name=task_name,
        lstm_model=lstm_model,
        xgb_model=xgb_model,
        lgbm_model=lgbm_model,
        ridge_model=ridge_meta,
        lstm_feature_scaler=lstm_feature_scaler,
        lstm_target_scaler=lstm_target_scaler,
        gbm_scaler=gbm_scaler,
        metadata=metadata,
    )

    return {
        "task": task_name,
        "results_df": results_df,
        "y_test": y_test,
        "predictions": {
            "LSTM": pred_lstm,
            "XGBoost": pred_xgb,
            "LightGBM": pred_lgbm,
            "Stacked(Ridge)": pred_stack,
        },
        "dates_test": dates_test,
    }

## 7) Run All Experiments (WTI & Brent, 1-day & 7-day)

In [ ]:
EXPERIMENTS = [
    ("WTI_close", 1),
    ("WTI_close", 7),
    ("Brent_close", 1),
    ("Brent_close", 7),
]

all_runs = []
all_result_frames = []

for target_col, horizon in EXPERIMENTS:
    try:
        run_out = run_single_experiment(df_raw, target_col=target_col, horizon=horizon)
        all_runs.append(run_out)
        all_result_frames.append(run_out["results_df"])
    except Exception as exc:
        print(f"[RUN ERROR] target={target_col}, horizon={horizon}: {exc}")

if len(all_result_frames) == 0:
    raise RuntimeError("No experiment completed successfully. Check API keys/data availability.")

all_results = pd.concat(all_result_frames, ignore_index=True)

print("\nFinal Comparison Table (Required Format)")
final_comparison = all_results[["Model", "MAE", "RMSE", "MAPE", "Dir Accuracy"]].copy()
display(final_comparison)

print("\nDetailed Comparison by Target/Horizon")
detailed = all_results[["Target", "Horizon", "Model", "MAE", "RMSE", "MAPE", "R2", "Dir Accuracy"]].copy()
display(detailed.sort_values(["Target", "Horizon", "RMSE"]).reset_index(drop=True))

print("\nOverall Mean Performance Across Tasks")
overall = (
    all_results.groupby("Model", as_index=False)[["MAE", "RMSE", "MAPE", "R2", "Dir Accuracy"]]
    .mean()
    .sort_values("RMSE")
    .reset_index(drop=True)
)
display(overall)

print(f"\nModel artifacts saved in: {MODEL_DIR}")

## 8) Optional: Load Saved Models and Forecast Latest

> Demonstrates production-style model reload and latest inference from saved artifacts.

In [ ]:
def load_task_artifacts(task_name: str) -> dict:
    artifacts = {
        "lstm_model": tf.keras.models.load_model(os.path.join(MODEL_DIR, f"lstm_{task_name}.keras")),
        "xgb_model": joblib.load(os.path.join(MODEL_DIR, f"xgb_{task_name}.joblib")),
        "lgbm_model": joblib.load(os.path.join(MODEL_DIR, f"lgbm_{task_name}.joblib")),
        "ridge_model": joblib.load(os.path.join(MODEL_DIR, f"ridge_{task_name}.joblib")),
        "lstm_feature_scaler": joblib.load(os.path.join(MODEL_DIR, f"lstm_feature_scaler_{task_name}.joblib")),
        "lstm_target_scaler": joblib.load(os.path.join(MODEL_DIR, f"lstm_target_scaler_{task_name}.joblib")),
        "gbm_scaler": joblib.load(os.path.join(MODEL_DIR, f"gbm_scaler_{task_name}.joblib")),
        "metadata": joblib.load(os.path.join(MODEL_DIR, f"metadata_{task_name}.joblib")),
    }
    return artifacts


def forecast_latest(base_df: pd.DataFrame, target_col: str, horizon: int) -> dict:
    task_name = f"{target_col.lower()}_h{horizon}"
    art = load_task_artifacts(task_name)

    model_df = build_modeling_frame(base_df, target_col=target_col, horizon=horizon)
    views = make_supervised_views(model_df, target_col=target_col, sequence_length=SEQUENCE_LENGTH)

    X_seq_last = views["X_seq"][-1:]
    X_flat_last = views["X_flat"][-1:]
    base_last = float(views["base_price"][-1])
    origin_date = pd.Timestamp(views["origin_dates"][-1])
    target_date = origin_date + pd.Timedelta(days=horizon)

    X_seq_last_scaled = scale_lstm_sequences_transform(X_seq_last, art["lstm_feature_scaler"])
    pred_lstm_scaled = art["lstm_model"].predict(X_seq_last_scaled, verbose=0).ravel()
    pred_lstm = art["lstm_target_scaler"].inverse_transform(pred_lstm_scaled.reshape(-1, 1)).ravel()[0]

    X_flat_last_scaled = art["gbm_scaler"].transform(X_flat_last)
    pred_xgb = art["xgb_model"].predict(X_flat_last_scaled).ravel()[0]
    pred_lgbm = art["lgbm_model"].predict(X_flat_last_scaled).ravel()[0]

    level1 = np.array([[pred_lstm, pred_xgb, pred_lgbm]], dtype=np.float32)
    pred_stack = art["ridge_model"].predict(level1).ravel()[0]

    out = {
        "task": task_name,
        "origin_date": str(origin_date.date()),
        "target_date": str(target_date.date()),
        "base_price": base_last,
        "pred_lstm": float(pred_lstm),
        "pred_xgb": float(pred_xgb),
        "pred_lgbm": float(pred_lgbm),
        "pred_stacked": float(pred_stack),
    }
    return out


# Example latest forecasts from saved artifacts (run after training cell)
for t_col, hz in EXPERIMENTS:
    try:
        pred = forecast_latest(df_raw, target_col=t_col, horizon=hz)
        print(pred)
    except Exception as exc:
        print(f"[INFERENCE ERROR] {t_col}, h={hz}: {exc}")